# ASEADOS-SDN-IoT — Keşifsel Veri Analizi (EDA)

**Amaç:** Model eğitmeden **önce** veri setini tanımak.
Verinin boyutuna, sütunlarına, sınıflarına ve içindeki sorunlara bakacağız.

**Nasıl kullanılır:**
1. Bu defteri **VS Code** (Jupyter eklentisiyle) veya **Jupyter Notebook** ile aç.
2. Hücreleri **yukarıdan aşağıya**, tek tek çalıştır (Shift + Enter).
3. Her hücreden sonra **"Neye bakmalı?"** notunu oku. Bulduklarını en son hücreye yaz.

> Burada veriyi **değiştirmiyoruz**, sadece inceliyoruz. Temizleme işi ön işleme (preprocessing) defterinde yapılacak.

## 0. CSV dosyasını yükle

Bu sürüm **kendi bilgisayarında** çalışır (VS Code veya Jupyter Notebook ile).

1. CSV dosyasını bu defterle **aynı klasöre** koy. En kolayı bu.
2. Aşağıdaki `CSV_PATH` satırına dosyanın adını yaz.

Dosya başka bir klasördeyse tam yolunu yaz. Windows'ta yolun başına `r` koy:
`CSV_PATH = r'C:\Users\Dogukan\Downloads\dosya_adi.csv'`

(Tam yolu bulmak için: Dosya Gezgini'nde dosyaya **Shift + sağ tık → "Yol olarak kopyala"**.)

In [ ]:
# CSV dosyasını data/ klasörüne ASEADOS_SDN_IoT.csv adıyla koy.
# Notebook notebooks/ içinden çalıştığı için yol bir üst klasöre çıkar.
CSV_PATH = '../data/ASEADOS_SDN_IoT.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(CSV_PATH, low_memory=False)
df.columns = df.columns.str.strip()   # remove spaces around column names
print('Rows, columns:', df.shape)
df.head()

**Neye bakmalı?**
- Satır sayısı yaklaşık **457.044**, sütun sayısı yaklaşık **83–84** olmalı (83 özellik + etiket).
- Sayılar çok farklıysa yanlış dosyayı açmış olabilirsin.

## 1. Etiket (label) sütununu bul

Etiket, "doğru cevap" sütunudur: Benign, DoS, DDoS, Bot, Probe.
Bu hücre onu otomatik olarak bulur.

In [ ]:
candidates = [c for c in df.columns if c.lower() in ('label', 'class', 'attack', 'category')]
print('Possible label columns:', candidates)

LABEL = candidates[0] if candidates else df.columns[-1]   # if nothing found, use the last column
print('Using label column (kullanılan etiket):', LABEL)

## 2. Sütunlar ve veri tipleri

- **Sayısal (numeric):** süre, paket sayısı gibi sayılar. Modeller bunları doğrudan kullanabilir.
- **Metin (object/str):** IP adresi veya etiket gibi yazılar. Bunlar için özel işlem gerekir.

In [ ]:
print(df.dtypes.value_counts())
print()
text_cols = df.select_dtypes(exclude=np.number).columns.tolist()
print('Text (object) columns:', text_cols)

**Neye bakmalı?** Sütunların çoğu sayısal olmalı.
Metin sütunları genellikle **kimlik sütunlarıdır**: Flow ID, Src IP, Dst IP, Timestamp ve etiket.

## 3. Sınıf dağılımı

Her sınıfta kaç akış (flow) var? EDA'daki en önemli tablo budur.

In [ ]:
counts = df[LABEL].value_counts()
table = pd.DataFrame({'count': counts, 'percent (%)': (counts / len(df) * 100).round(2)})
print(table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Flows per class (normal scale)')
axes[0].set_ylabel('Number of flows')

counts.plot(kind='bar', ax=axes[1], color='darkorange', logy=True)
axes[1].set_title('Flows per class (log scale)')
axes[1].set_ylabel('Number of flows (log)')
plt.tight_layout()
plt.show()

**Neye bakmalı?**
- Beklenen (README'ye göre): Benign ~%57, DoS ~%28, DDoS ~%11, Bot ~%2, Probe ~%1,7.
- Normal grafikte Bot ve Probe neredeyse görünmez. Buna **sınıf dengesizliği (class imbalance)** denir.
- **Logaritmik ölçekli** grafik küçük sınıfları görünür hale getirir.
- Etiketlerin **tam yazılışını** not al ("Bot" mu "Botnet" mi, "BENIGN" mi "Benign" mı). Sonraki kodlarda lazım olacak.

## 4. Eksik ve sonsuz değerler

- **NaN (eksik):** boş hücre.
- **inf (sonsuz):** CICFlowMeter sıfıra bölme olduğunda `inf` üretir (örneğin süresi 0 olan bir akışın "saniyedeki paket sayısı").

Modeller ikisinde de hata verir, bu yüzden bulmamız gerekiyor.

In [ ]:
num = df.select_dtypes(include=np.number)

nan_per_col = df.isna().sum()
inf_per_col = np.isinf(num).sum()

print('Total NaN cells :', int(nan_per_col.sum()))
print('Total inf cells :', int(inf_per_col.sum()))
print()
print('Columns with NaN:')
print(nan_per_col[nan_per_col > 0])
print()
print('Columns with inf:')
print(inf_per_col[inf_per_col > 0])

bad_rows = (df.isna().any(axis=1) | np.isinf(num).any(axis=1))
print()
print(f'Rows with at least one NaN or inf: {bad_rows.sum()} ({bad_rows.mean()*100:.2f}% of all rows)')

**Neye bakmalı?**
- Hangi sütunlarda sorun var? Genellikle "Flow Bytes/s" ve "Flow Packets/s".
- Bozuk satırların **yüzdesi** ne kadar? Küçükse (örneğin %1'den az) bu satırları sonra **silebiliriz**.

## 5. Tekrarlı satırlar (duplicates)

Aynı satırın çok kez olması modelin ezberlemesine yol açabilir. Ayrıca aynı satır hem train hem test kümesine düşebilir. Bu da skoru gerçekte olduğundan daha iyi gösterir.

In [ ]:
dup = df.duplicated()
print(f'Duplicate rows: {dup.sum()} ({dup.mean()*100:.2f}%)')
print()
print('Duplicates per class (sınıf bazında tekrar):')
print(df[dup][LABEL].value_counts())

**Neye bakmalı?** En çok tekrar hangi sınıfta? Saldırı trafiğinde (DoS gibi) birebir aynı akışlar çok olur.
Not: Tekrarları silersen sınıf sayıları değişir. Yeni sayıları rapora yaz.

## 6. Sabit sütunlar

**Her satırda aynı değere sahip** bir sütun modele hiçbir bilgi vermez. Bunu silebiliriz.

In [ ]:
nunique = df.nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
print('Number of constant columns:', len(constant_cols))
print(constant_cols)

## 7. Kimlik sütunları — veri sızıntısı (data leakage) riski

Flow ID, IP adresleri, portlar ve Timestamp gibi sütunlar trafiğin **nasıl davrandığını** değil, **kimden geldiğini** söyler.
Bunları tutarsak model "bu IP = saldırgan" diye ezberleyebilir. Buna **veri sızıntısı (data leakage)** denir.

Bu hücre, her birinin etiketle ne kadar güçlü bağlantılı olduğunu gösterir.

In [ ]:
id_keywords = ['flow id', 'src ip', 'dst ip', 'source ip', 'destination ip',
               'src port', 'dst port', 'source port', 'destination port', 'timestamp']
id_cols = [c for c in df.columns if c.lower() in id_keywords]
print('Identifier columns found:', id_cols)
print()

for col in id_cols:
    if col.lower() == 'flow id':
        continue   # almost every value is unique, nothing to see here
    top = df[col].value_counts().head(5).index
    print(f'--- {col}: label share for its 5 most common values ---')
    print(pd.crosstab(df[col][df[col].isin(top)], df[LABEL], normalize='index').round(2))
    print()

**Neye bakmalı?** Bir IP (örneğin Kali saldırgan makinesi) neredeyse %100 tek bir saldırı sınıfına aitse, model onunla kopya çekebilir.
Bu, raporda "bu sütunları neden sildik?" sorusunun kanıtı olur.

Not: **Destination port (hedef port)** özel bir durum. Bazen gerçek bir özellik olarak tutulur (örneğin port 80 = web trafiği). Grupta tartışın.

## 8. Temel istatistikler

Her sayısal sütunun en küçük (min), en büyük (max), ortalama (mean) ve ortanca (median, 50%) değerleri.

In [ ]:
desc = num.replace([np.inf, -np.inf], np.nan).describe().T
desc[['mean', 'std', 'min', '50%', 'max']]

**Neye bakmalı?**
- Ortancaya göre çok büyük **max** değerleri → **aykırı değerler (outliers)** ve çok farklı ölçekler.
- Ölçekleri çok farklı sütunlar (örneğin 0–1 ile 0–100.000.000) Logistic Regression için **ölçekleme (scaling)** gerektirir. Ağaç modelleri (Random Forest, LightGBM) gerektirmez.
- Süre veya sayı sütunlarında **negatif değer** fiziksel olarak mümkün değil → muhtemel gürültü (noise).

## 9. Sınıfları birkaç özellikte karşılaştır

Saldırılar normal trafikten farklı görünüyor mu? Görünüyorsa model bunu öğrenebilir.
Değerler çok yayıldığı için kutu grafiklerini (boxplot) **logaritmik ölçekle** çiziyoruz.

In [ ]:
wanted = ['Flow Duration', 'Tot Fwd Pkts', 'Flow Byts/s', 'Flow Pkts/s',
          'Fwd Pkt Len Mean', 'Flow IAT Mean']
features = [c for c in wanted if c in df.columns][:4]
if len(features) < 4:   # if names are different, take the first numeric columns
    features += [c for c in num.columns if c not in features][:4 - len(features)]
print('Plotting:', features)

sample = pd.concat([g.sample(min(len(g), 3000), random_state=42) for _, g in df.groupby(LABEL)])   # max 3000 rows per class

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.ravel(), features):
    data = sample[[col, LABEL]].replace([np.inf, -np.inf], np.nan).dropna()
    data[col] = data[col].clip(lower=0) + 1   # +1 so log scale works with zeros
    sns.boxplot(data=data, x=LABEL, y=col, ax=ax, showfliers=False)
    ax.set_yscale('log')
    ax.set_title(col)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

**Neye bakmalı?**
- İki sınıfın kutuları **farklı yüksekliklerdeyse**, o özellik bu sınıfları iyi ayırıyor demektir.
- DoS ve DDoS kutuları **neredeyse aynıysa**, model bunları karıştırabilir. Bunu karışıklık matrisi (confusion matrix) için aklında tut.
- Makale, Benign ile saldırıların "açıkça ayrılabildiğini" söylüyor. Bunu burada görüyor musun?

## 10. Yüksek korelasyonlu özellikler

Korelasyonu 1'e yakın iki sütun **aynı bilgiyi** taşır (örneğin "toplam paket" ve "toplam byte").
Ağaç modelleri için büyük sorun değil, ama bilmek ve rapora yazmak iyi olur.

In [ ]:
clean_num = num.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how='all')
clean_num = clean_num.loc[:, clean_num.nunique() > 1]
corr = clean_num.sample(min(len(clean_num), 50000), random_state=42).corr().abs()

upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack().sort_values(ascending=False)
print('Pairs with correlation > 0.95:', int((pairs > 0.95).sum()))
print(pairs.head(15).round(3))

top = pd.Index(pairs.head(15).index.get_level_values(0)).append(pd.Index(pairs.head(15).index.get_level_values(1))).unique()[:15]
plt.figure(figsize=(10, 8))
sns.heatmap(corr.loc[top, top], cmap='viridis', vmin=0, vmax=1)
plt.title('Correlation of the most correlated features')
plt.tight_layout()
plt.show()

## 11. Bulguların ✍️

Cevapları kendi cümlelerinle yaz. Bunları raporda ve ön işleme adımında kullanacaksın.

1. Tam satır ve sütun sayısı:
2. Etiketlerin tam yazılışı ve sınıf yüzdeleri:
3. NaN / inf olan sütunlar ve bozuk satırların yüzdesi:
4. Tekrarlı satır sayısı (en çok hangi sınıfta):
5. Silinecek sabit sütunlar:
6. Silinecek kimlik sütunları (Destination Port'u tutacak mıyız?):
7. Sınıfları iyi ayıran özellikler:
8. Birbirine benzeyen sınıflar (muhtemel karışıklık):
9. Garip veya şaşırtıcı bir şey:

## 12. Ek kontroller (Faz 2'den önce)

Negatif Flow Duration, Protocol–etiket tablosu ve kimlik sütunları olmadan tekrar eden satırlar.

In [ ]:
# 1) Negative values in duration
print('Negative Flow Duration rows:', (df['Flow Duration'] < 0).sum())

# 2) Protocol vs label: is Protocol also a shortcut?
print(pd.crosstab(df['Protocol'], df[LABEL]))

# 3) Duplicates after removing ID columns
id_cols = ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Timestamp']
print('Duplicates without ID columns:', df.drop(columns=id_cols).duplicated().sum())